# 10 Bootstrap Ci

**Purpose.** Bootstrap confidence intervals on out-of-fold predictions.

**Thesis reference.** §4.7 / §5

**Inputs / outputs.** See the data-layout table in the repository README. Paths near the top of the notebook (e.g. `PROFILE_DIR`, `IMAGE_ROOT`, `LABELS`) are relative to the repository root and assume the folder layout described there.

---

In [4]:

import json
import numpy as np
from pathlib import Path
from scipy.stats import mannwhitneyu
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import RepeatedStratifiedKFold

# CONFIG
PROFILES   = "profiles"
MAXF       = 3000
N_REPEATS  = 10          # 5-fold x this many = out-of-fold predictions
N_BOOT     = 2000

# (label file, which representation to use)
JOBS = [
    ("labels/labels_severity_nondiabetic.json",   "profile_1d"),
    ("labels/labels_positive_control_diabetes.json", "map_2d"),
    ("labels/labels_severity_all.json",           "map_2d"),
    ("labels/labels_dkd_vs_nondiabetic_mckd.json", "profile_1d"),
    ("labels/labels_detect_any_ckd.json",         "map_2d"),
    ("labels/labels_matched_nondiabetic.json",    "profile_1d"),
    ("labels/labels_progression.json",               "map_2d"),

]

ALL_PIDS = json.load(open(f"{PROFILES}/profile_pids.json"))
_F = _C = None

def features(kind, idx):
    """Build the requested representation, TIC-normalised and log-scaled."""
    global _F, _C
    if kind in ("profile_1d", "intensity_stats"):
        if _F is None:
            _F = np.load(f"{PROFILES}/mz_profiles.npy")
        M = _F[idx].astype(np.float64)
    else:
        if _C is None:
            _C = np.load(f"{PROFILES}/coarse_maps.npy")
        C = _C[idx].astype(np.float64)
        M = C.sum(axis=2) if kind == "rt_profile" else C.reshape(len(C), -1)
    t = M.sum(1, keepdims=True)
    t[t == 0] = 1
    X = np.log1p(M / t * 1e6)
    if X.shape[1] > MAXF:
        X = X[:, np.sort(np.argsort(X.var(0))[::-1][:MAXF])]
    return X

def make_model():
    return Pipeline([("s", StandardScaler()),
                     ("c", LogisticRegression(C=0.1, max_iter=3000,
                                              class_weight="balanced"))])

def run(label_file, kind):
    p = Path(label_file)
    if not p.exists():
        print(f"[skip] {p.name} not found\n")
        return None
    lab = json.load(open(p)).get("labels")
    idx = [i for i, q in enumerate(ALL_PIDS) if q in lab]
    if len(idx) < 40:
        print(f"[skip] {p.name}: only {len(idx)} patients with data\n")
        return None
    y = np.array([lab[ALL_PIDS[i]] for i in idx])
    if min(np.bincount(y)) < 15:
        print(f"[skip] {p.name}: too few in one class\n")
        return None

    # patient-ID baseline for THIS cohort, computed not assumed
    ids = np.array([int("".join(c for c in ALL_PIDS[i] if c.isdigit()))
                    for i in idx])
    u, _ = mannwhitneyu(ids[y == 1], ids[y == 0])
    id_auc = u / ((y == 1).sum() * (y == 0).sum())
    id_auc = max(id_auc, 1 - id_auc)

    X = features(kind, idx)
    oof = np.zeros(len(y)); cnt = np.zeros(len(y))
    for tr, te in RepeatedStratifiedKFold(n_splits=5, n_repeats=N_REPEATS,
                                          random_state=42).split(X, y):
        oof[te] += make_model().fit(X[tr], y[tr]).predict_proba(X[te])[:, 1]
        cnt[te] += 1
    oof /= cnt
    auc = roc_auc_score(y, oof)

    rng = np.random.default_rng(0)
    bs = []
    for _ in range(N_BOOT):
        i = rng.choice(len(y), len(y), replace=True)
        if len(np.unique(y[i])) > 1:
            bs.append(roc_auc_score(y[i], oof[i]))
    lo, hi = np.percentile(bs, [2.5, 97.5])

    task = p.stem.replace("labels_", "")
    print(f"=== {task}  ({kind}) ===")
    print(f"  n = {len(y)}  ({int(y.sum())} positive / {int((1-y).sum())} negative)")
    print(f"  AUC          {auc:.4f}   95% CI [{lo:.4f}, {hi:.4f}]")
    print(f"  patient ID   {id_auc:.4f}   <- baseline for this cohort")
    print(f"  margin       {auc-id_auc:+.4f}")
    print(f"  CI excludes 0.5           : {'YES' if lo > 0.5 else 'no'}")
    print(f"  CI excludes the ID baseline: {'YES' if lo > id_auc else 'no'}")
    print()
    return dict(task=task, features=kind, n=int(len(y)), auc=float(auc),
                ci_lo=float(lo), ci_hi=float(hi), id_auc=float(id_auc),
                margin=float(auc - id_auc),
                beats_id=bool(lo > id_auc), above_chance=bool(lo > 0.5))

rows = [r for j in JOBS if (r := run(*j)) is not None]

print("=" * 78)
print(f"{'task':30s} {'n':>4} {'AUC':>7} {'95% CI':>18} {'ID':>7} {'beats ID':>9}")
print("-" * 78)
for r in sorted(rows, key=lambda r: -r["margin"]):
    ci = f"[{r['ci_lo']:.3f}, {r['ci_hi']:.3f}]"
    print(f"{r['task']:30s} {r['n']:4d} {r['auc']:7.4f} {ci:>18s} "
          f"{r['id_auc']:7.3f} {'YES' if r['beats_id'] else 'no':>9}")

with open("bootstrap_ci.json", "w") as fh:
    json.dump(rows, fh, indent=2)
print("\nWritten to bootstrap_ci.json")

=== severity_nondiabetic  (profile_1d) ===
  n = 359  (66 positive / 293 negative)
  AUC          0.6407   95% CI [0.5625, 0.7155]
  patient ID   0.5187   <- baseline for this cohort
  margin       +0.1220
  CI excludes 0.5           : YES
  CI excludes the ID baseline: YES

=== positive_control_diabetes  (map_2d) ===
  n = 596  (191 positive / 405 negative)
  AUC          0.7753   95% CI [0.7306, 0.8147]
  patient ID   0.6431   <- baseline for this cohort
  margin       +0.1323
  CI excludes 0.5           : YES
  CI excludes the ID baseline: YES

=== severity_all  (map_2d) ===
  n = 510  (121 positive / 389 negative)
  AUC          0.7410   95% CI [0.6902, 0.7907]
  patient ID   0.5352   <- baseline for this cohort
  margin       +0.2058
  CI excludes 0.5           : YES
  CI excludes the ID baseline: YES

=== dkd_vs_nondiabetic_mckd  (profile_1d) ===
  n = 121  (55 positive / 66 negative)
  AUC          0.6950   95% CI [0.6044, 0.7849]
  patient ID   0.5650   <- baseline for this coh

In [3]:
import json
import numpy as np
from pathlib import Path
from scipy.stats import spearmanr
from scipy.spatial.distance import pdist

EMB_DIR = "embeddings"
LABELS  = "labels/labels_severity_nondiabetic.json"
lab = json.load(open(LABELS))["labels"]

reps = {}
for f in sorted(Path(EMB_DIR).glob("*.npy")):
    meta = f.with_suffix(".json")
    if not meta.exists():
        continue
    info = json.load(open(meta))
    keep = [i for i, p in enumerate(info["pids"]) if p in lab]
    if len(keep) < 30:
        continue
    reps[f"{info['representation']}/{info['model']}"] = (
        np.load(f)[keep],
        np.array([lab[info["pids"][i]] for i in keep]))

print(f"loaded {len(reps)} representations, "
      f"{len(next(iter(reps.values()))[1])} patients\n")

# RSA
if len(reps) > 1:
    names = list(reps)
    rdms = {}
    for n in names:
        X = reps[n][0]
        Xz = (X - X.mean(0)) / (X.std(0) + 1e-9)
        rdms[n] = pdist(Xz, metric="correlation")

    print("REPRESENTATIONAL SIMILARITY  (Spearman between distance matrices)")
    hdr = "".join(f"{n.split('/')[1][:9]:>11s}" for n in names)
    print(f"{'':32s}{hdr}")
    for a in names:
        row = f"{a[:31]:32s}"
        for b in names:
            r = 1.0 if a == b else spearmanr(rdms[a], rdms[b])[0]
            row += f"{r:11.3f}"
        print(row)

    off = [spearmanr(rdms[a], rdms[b])[0]
           for i, a in enumerate(names) for b in names[i+1:]]
    m = float(np.mean(off))
    print(f"\nmean off-diagonal similarity: {m:.3f}")
    if m > 0.7:
        print("  -> encoders organise patients very similarly; the ensemble of")
        print("     representations in the original protocol added little.")
    elif m > 0.4:
        print("  -> partial overlap; representations share much but not all.")
    else:
        print("  -> representations capture substantially different information.")

# SHAP via XGBoost
print("\n" + "=" * 62)
try:
    import xgboost as xgb
    from sklearn.model_selection import train_test_split
    from sklearn.metrics import roc_auc_score

    best = max(reps, key=lambda k: reps[k][0].shape[1])
    X, y = reps[best]
    print(f"TreeSHAP on {best}   X = {X.shape}")

    Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.3,
                                          stratify=y, random_state=42)
    spw = (ytr == 0).sum() / max((ytr == 1).sum(), 1)
    m = xgb.XGBClassifier(n_estimators=200, max_depth=3, learning_rate=0.05,
                          subsample=0.8, eval_metric="auc",
                          scale_pos_weight=spw)
    m.fit(Xtr, ytr)
    print(f"  held-out AUC: {roc_auc_score(yte, m.predict_proba(Xte)[:,1]):.4f}")

    # exact TreeSHAP; last column is the bias term
    contribs = m.get_booster().predict(xgb.DMatrix(Xte), pred_contribs=True)
    sv = contribs[:, :-1]
    imp = np.abs(sv).mean(0)
    order = np.argsort(imp)[::-1]
    tot = imp.sum()

    print(f"\ntop 15 embedding dimensions by mean |SHAP|:")
    for r, i in enumerate(order[:15], 1):
        print(f"  {r:2d}. dim {i:5d}   {imp[i]:.5f}   "
              f"({imp[i]/tot:5.2%} of total)")

    print()
    for k in (10, 50, 100, 500):
        if k <= len(imp):
            print(f"  top {k:4d} of {len(imp)} dims carry "
                  f"{imp[order[:k]].sum()/tot:6.1%} of total attribution")

    nz = (imp > 0).sum()
    print(f"\n  {nz} of {len(imp)} dimensions received non-zero attribution")
    if imp[order[:10]].sum() / tot > 0.5:
        print("  -> attribution is CONCENTRATED: a few dimensions dominate.")
    else:
        print("  -> attribution is DIFFUSE, consistent with the region-discovery")
        print("     finding that the signal is distributed rather than localised.")
except ImportError:
    print("xgboost not available in this kernel.")
except Exception as e:
    print(f"failed: {type(e).__name__}: {e}")


loaded 6 representations, 359 patients

REPRESENTATIONAL SIMILARITY  (Spearman between distance matrices)
                                  efficient   resnet18   vit_b_16  efficient   resnet18   vit_b_16
grayscale_heatmap/efficientnet_       1.000      0.751      0.678      0.743      0.689      0.543
grayscale_heatmap/resnet18            0.751      1.000      0.632      0.696      0.675      0.554
grayscale_heatmap/vit_b_16            0.678      0.632      1.000      0.640      0.569      0.507
rgb_mz_split/efficientnet_b0          0.743      0.696      0.640      1.000      0.765      0.601
rgb_mz_split/resnet18                 0.689      0.675      0.569      0.765      1.000      0.582
rgb_mz_split/vit_b_16                 0.543      0.554      0.507      0.601      0.582      1.000

mean off-diagonal similarity: 0.642
  -> partial overlap; representations share much but not all.

TreeSHAP on grayscale_heatmap/efficientnet_b0   X = (359, 1280)
  held-out AUC: 0.5648

top 15 embedd